In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# VOCAB
# ======================
CHARS = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
char2idx = {c: i+1 for i, c in enumerate(CHARS)}
idx2char = {i+1: c for i, c in enumerate(CHARS)}
idx2char[0] = ""
NUM_CLASSES = len(CHARS) + 1



In [ ]:
class CaptchaDataset(Dataset):
    def __init__(self, img_dir):
        self.img_dir = img_dir
        self.images = [
            f for f in os.listdir(img_dir)
            if f.lower().endswith(tuple([".png", ".jpg", ".jpeg"]))
        ]

        self.transform = transforms.Compose([
            transforms.Grayscale(),
            transforms.Resize((100, 200)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]

        # 🔥 IMPORTANT FIX
        label = img_name.split("_")[0]

        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("L")
        image = self.transform(image)

        # Filter characters to ensure they are in char2idx
        filtered_label = "".join([c for c in label if c in char2idx])
        label_idx = torch.tensor(
            [char2idx[c] for c in filtered_label],
            dtype=torch.long
        )

        return image, label_idx

In [ ]:
def collate_fn(batch):
    images, labels = zip(*batch)

    images = torch.stack(images)
    label_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    labels = torch.cat(labels)

    input_lengths = torch.full(
        size=(images.size(0),),
        fill_value=7,   # Corrected CNN output width from 50 to 7
        dtype=torch.long
    )

    return images, labels, input_lengths, label_lengths

In [ ]:
class CRNN(nn.Module):
    def __init__(self):
        super().__init__()

        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        # Modify the first conv layer to accept 1 input channel (grayscale)
        resnet.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

        self.cnn = nn.Sequential(*list(resnet.children())[:-2])

        self.conv1x1 = nn.Conv2d(512, 256, 1)

        self.lstm = nn.LSTM(
            input_size=256 * 4,
            hidden_size=256,
            num_layers=2,
            bidirectional=True
        )

        self.fc = nn.Linear(512, NUM_CLASSES)

    def forward(self, x):
        x = self.cnn(x)           # (B,512,4,50)
        x = self.conv1x1(x)       # (B,256,4,50)

        b, c, h, w = x.size()
        x = x.permute(3, 0, 1, 2).contiguous()
        x = x.view(w, b, c*h)     # (50,B,1024)

        x, _ = self.lstm(x)
        x = self.fc(x)
        return x

In [ ]:
train_ds = CaptchaDataset("/content/drive/MyDrive/precog_captcha/task1/train")
val_ds   = CaptchaDataset("/content/drive/MyDrive/precog_captcha/task1/validation")
test_ds  = CaptchaDataset("/content/drive/MyDrive/precog_captcha/task1/test")


train_loader = DataLoader(
    train_ds, batch_size=32,
    shuffle=True, collate_fn=collate_fn
)

val_loader = DataLoader(
    val_ds, batch_size=32,
    shuffle=False, collate_fn=collate_fn
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CRNN().to(device)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
EPOCHS = 30
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, labels, in_len, tgt_len in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        log_probs = outputs.log_softmax(2)

        loss = criterion(log_probs, labels, in_len, tgt_len)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")


In [ ]:
def decode(pred):
    pred = pred.argmax(2)
    prev = -1
    text = ""

    for p in pred:
        p = p.item()
        if p != prev and p != 0:
            text += idx2char[p]
        prev = p
    return text


In [ ]:
model.eval()
with torch.no_grad():
    for images, _, _, _ in val_loader:
        images = images.to(device)
        out = model(images)
        print("Prediction:", decode(out[:,0,:]))
        break
